In [ ]:
import gzip
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re
import zipfile
from scipy.sparse import coo_matrix
from sklearn.neighbors import NearestNeighbors

In [ ]:
!wc -l goodreads_books.json.gz

7617498 goodreads_books.json.gz


In [ ]:
!ls -lh | grep goodreads_books

-rw-r--r-- 1 root root 2.0G May 14 08:39 goodreads_books.json.gz


In [ ]:
import gzip

In [ ]:
with gzip.open("goodreads_books.json.gz",'r') as f:
    line = f.readline()

In [ ]:
line

b'{"isbn": "0312853122", "text_reviews_count": "1", "series": [], "country_code": "US", "language_code": "", "popular_shelves": [{"count": "3", "name": "to-read"}, {"count": "1", "name": "p"}, {"count": "1", "name": "collection"}, {"count": "1", "name": "w-c-fields"}, {"count": "1", "name": "biography"}], "asin": "", "is_ebook": "false", "average_rating": "4.00", "kindle_asin": "", "similar_books": [], "description": "", "format": "Paperback", "link": "https://www.goodreads.com/book/show/5333265-w-c-fields", "authors": [{"author_id": "604031", "role": ""}], "publisher": "St. Martin\'s Press", "num_pages": "256", "publication_day": "1", "isbn13": "9780312853129", "publication_month": "9", "edition_information": "", "publication_year": "1984", "url": "https://www.goodreads.com/book/show/5333265-w-c-fields", "image_url": "https://images.gr-assets.com/books/1310220028m/5333265.jpg", "book_id": "5333265", "ratings_count": "3", "work_id": "5400751", "title": "W.C. Fields: A Life on Film", "t

In [ ]:
import json

In [ ]:
json.loads(line)

{'isbn': '0312853122',
 'text_reviews_count': '1',
 'series': [],
 'country_code': 'US',
 'language_code': '',
 'popular_shelves': [{'count': '3', 'name': 'to-read'},
  {'count': '1', 'name': 'p'},
  {'count': '1', 'name': 'collection'},
  {'count': '1', 'name': 'w-c-fields'},
  {'count': '1', 'name': 'biography'}],
 'asin': '',
 'is_ebook': 'false',
 'average_rating': '4.00',
 'kindle_asin': '',
 'similar_books': [],
 'description': '',
 'format': 'Paperback',
 'link': 'https://www.goodreads.com/book/show/5333265-w-c-fields',
 'authors': [{'author_id': '604031', 'role': ''}],
 'publisher': "St. Martin's Press",
 'num_pages': '256',
 'publication_day': '1',
 'isbn13': '9780312853129',
 'publication_month': '9',
 'edition_information': '',
 'publication_year': '1984',
 'url': 'https://www.goodreads.com/book/show/5333265-w-c-fields',
 'image_url': 'https://images.gr-assets.com/books/1310220028m/5333265.jpg',
 'book_id': '5333265',
 'ratings_count': '3',
 'work_id': '5400751',
 'title': '

In [ ]:
def parse_fields(line):
    data = json.loads(line)
    return {
        "book_id": data["book_id"],
        "title": data["title"],
        "average_rating": data["average_rating"],
        "ratings_count": data["ratings_count"],
        "url": data["url"],
        "cover_image": data["image_url"]
    }

In [ ]:
books = []
with gzip.open("goodreads_books.json.gz") as f:
    while True:
        line = f.readline()
        if not line:
            break
        fields = parse_fields(line)

        try:
            ratings_count = int(fields["ratings_count"])
            average_rating = float(fields["average_rating"])
        except ValueError:
            continue
        if ratings_count >= 100 and average_rating >= 3:
          books.append(fields)

In [ ]:
titles = pd.DataFrame.from_dict(books)
titles

,book_id,title,average_rating,ratings_count,url,cover_image
0,7327624,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",4.03,140,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...
1,6066819,Best Friends Forever,3.49,51184,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...
2,287149,The Devil's Notebook,3.81,986,https://www.goodreads.com/book/show/287149.The...,https://images.gr-assets.com/books/1328768789m...
3,6066814,"Crowner Royal (Crowner John Mystery, #13)",3.93,186,https://www.goodreads.com/book/show/6066814-cr...,https://images.gr-assets.com/books/1328724803m...
4,33394837,The House of Memory (Pluto's Snitch #2),4.33,269,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...
...,...,...,...,...,...,...
496956,15500943,"Not Quickly Broken (Chop, Chop, #7)",4.52,456,https://www.goodreads.com/book/show/15500943-n...,https://images.gr-assets.com/books/1381765124m...
496957,15734522,Why You're Not Married . . . Yet: The Straight...,3.79,101,https://www.goodreads.com/book/show/15734522-w...,https://images.gr-assets.com/books/1341351347m...
496958,1370179,The Brazilian Boss's Innocent Mistress,3.51,240,https://www.goodreads.com/book/show/1370179.Th...,https://s.gr-assets.com/assets/nophoto/book/11...
496959,17805813,"Ondine (Ondine Quartet, #0.5)",4.02,327,https://www.goodreads.com/book/show/17805813-o...,https://images.gr-assets.com/books/1379766592m...


In [ ]:
titles["ratings_count"] = pd.to_numeric(titles["ratings_count"])
titles["average_rating"] = pd.to_numeric(titles["average_rating"])
titles['book_id'] = titles['book_id'].astype(str)

In [ ]:
titles["mod_title"] = titles["title"].str.replace("[^a-zA-Z0-9 ]", "", regex=True).str.lower()

In [ ]:
titles["mod_title"] = titles["mod_title"].str.replace("\s+", " ", regex=True)

In [ ]:
titles = titles[titles["mod_title"].str.len() > 0]

In [ ]:
titles.head()

,book_id,title,average_rating,ratings_count,url,cover_image,mod_title
0,7327624,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",4.03,140,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,the unschooled wizard sun wolf and starhawk 12
1,6066819,Best Friends Forever,3.49,51184,https://www.goodreads.com/book/show/6066819-be...,https://s.gr-assets.com/assets/nophoto/book/11...,best friends forever
2,287149,The Devil's Notebook,3.81,986,https://www.goodreads.com/book/show/287149.The...,https://images.gr-assets.com/books/1328768789m...,the devils notebook
3,6066814,"Crowner Royal (Crowner John Mystery, #13)",3.93,186,https://www.goodreads.com/book/show/6066814-cr...,https://images.gr-assets.com/books/1328724803m...,crowner royal crowner john mystery 13
4,33394837,The House of Memory (Pluto's Snitch #2),4.33,269,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,the house of memory plutos snitch 2


In [ ]:
len(titles["book_id"].unique())

495230

In [ ]:
vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform(titles["mod_title"])

In [ ]:
def search(query, vectorizer):
    processed = re.sub("[^a-zA-Z0-9 ]", "", query.lower())
    query_vec = vectorizer.transform([processed])
    similarity = cosine_similarity(query_vec, tfidf).flatten()
    indices = np.argpartition(similarity, -20)[-20:]
    sorted_indices = indices[np.argsort(similarity[indices])[::-1]]
    results = titles.iloc[sorted_indices].copy()
    results['similarity_score'] = similarity[sorted_indices]
    results['ratings_count'] = results['ratings_count'].astype(int)
    results = results.sort_values(by='ratings_count', ascending=False)
    return results.head(5)

In [ ]:
search("Battle Cry of Freedom", vectorizer)

,book_id,title,average_rating,ratings_count,url,cover_image,mod_title,similarity_score
407772,35100,Battle Cry of Freedom,4.32,19595,https://www.goodreads.com/book/show/35100.Batt...,https://s.gr-assets.com/assets/nophoto/book/11...,battle cry of freedom,1.000000
323593,42694,Battle Cry,4.14,7536,https://www.goodreads.com/book/show/42694.Batt...,https://images.gr-assets.com/books/1410141181m...,battle cry,0.801576
261453,8474899,Freedom,3.73,3971,https://www.goodreads.com/book/show/8474899-fr...,https://images.gr-assets.com/books/1327935005m...,freedom,0.564754
39225,17846926,The Cry,3.74,1404,https://www.goodreads.com/book/show/17846926-t...,https://images.gr-assets.com/books/1375213561m...,the cry,0.577338
408558,283636,Battle Cry (#2),3.81,725,https://www.goodreads.com/book/show/283636.Bat...,https://images.gr-assets.com/books/1298858353m...,battle cry 2,0.801576


In [ ]:
print(titles[titles["book_id"] == "18373318"])

         book_id                        title  average_rating  ratings_count  \
241422  18373318  Earth 2, Vol. 3: Battle Cry            3.69            506   

                                                      url  \
241422  https://www.goodreads.com/book/show/18373318-e...   

                                              cover_image  \
241422  https://images.gr-assets.com/books/1393623291m...   

                       mod_title  
241422  earth 2 vol 3 battle cry  


In [ ]:
def create_my_books_df(my_titles_list, tfidf_vectorizer):
  my_books = pd.DataFrame()
  for title, rating in my_titles_list:
    results = search(title, tfidf_vectorizer)
    if not results.empty:
      results = results[["book_id", "title"]].copy()
      results["rating"] = rating
      results["user_id"] = "-1"
      my_books = pd.concat([my_books, results[["user_id","book_id","title","rating"]]])

    my_books = my_books.reset_index(drop=True)
    my_books['user_id'] = my_books['user_id'].astype(str)
    my_books['book_id'] = my_books['book_id'].astype(str)
    my_books["rating"] = pd.to_numeric(my_books["rating"])

  return my_books

In [ ]:
my_title_list_romance = [
    ("Twilight", 5),
    ("New Moon", 4),
    ("Eclipse", 4),
    ("Breaking Dawn", 4),
    ("Fifty Shades of Grey", 3),
    ("Fifty Shades Darker", 3),
    ("Fifty Shades Freed", 3),
    ("Pride and Prejudice", 5),
    ("Emma", 4),
    ("Sense and Sensibility", 4),
    ("Me Before You", 5),
    ("After", 3),
    ("Beautiful Disaster", 3),
    ("It Ends with Us", 5),
    ("Ugly Love", 4),
    ("November 9", 4),
    ("Confess", 4),
    ("Verity", 4),
    ("Reminders of Him", 4),
    ("The Notebook", 5),
    ("Dear John", 4),
    ("The Last Song", 4),
    ("Safe Haven", 4),
    ("The Lucky One", 4),
    ("The Best of Me", 4),
    ("The Wedding", 3),
    ("The Choice", 3),
    ("To All the Boys I've Loved Before", 4),
    ("P.S. I Still Love You", 4),
    ("Always and Forever, Lara Jean", 4),
    ("The Kiss Quotient", 5),
    ("The Bride Test", 4),
    ("Red, White & Royal Blue", 5),
    ("Beach Read", 4),
    ("People We Meet on Vacation", 4),
    ("Book Lovers", 4),
    ("Love and Other Words", 4),
    ("One Day", 4),
    ("Outlander", 5),
    ("Dragonfly in Amber", 4),
    ("Voyager", 4),
    ("The Bronze Horseman", 5),
    ("Tatiana and Alexander", 4),
    ("The Summer Garden", 4),
    ("Beautiful Disaster", 3),
    ("Walking Disaster", 3),
    ("Thoughtless", 3),
    ("Slammed", 4),
    ("Point of Retreat", 4),
    ("Hopeless", 4)
]


In [ ]:
my_title_list_nonfiction = [
    ("The Smartest Guys in the Room", 5),
    ("Endurance", 4),
    ("The Mask of Command", 4),
    ("Battle Cry of Freedom", 5),
    ("Sapiens", 5),
    ("Homo Deus", 4),
    ("Educated", 5),
    ("Becoming", 5),
    ("Thinking, Fast and Slow", 5),
    ("Guns, Germs, and Steel", 5),
    ("The Immortal Life of Henrietta Lacks", 5),
    ("Quiet", 4),
    ("Outliers", 4),
    ("Atomic Habits", 5),
    ("The Power of Habit", 4),
    ("Can't Hurt Me", 5),
    ("12 Rules for Life", 4),
    ("Man's Search for Meaning", 5),
    ("The Subtle Art of Not Giving a F*ck", 4),
    ("The Psychology of Money", 4),
    ("Dare to Lead", 4),
    ("Grit", 4),
    ("Start with Why", 4),
    ("Extreme Ownership", 4),
    ("The Lean Startup", 4),
    ("Shoe Dog", 5),
    ("The Everything Store", 4),
    ("Elon Musk: Tesla, SpaceX, and the Quest for a Fantastic Future", 5),
    ("Steve Jobs", 5),
    ("Bad Blood", 4),
    ("The Big Short", 5),
    ("Thinking in Bets", 4),
    ("Measure What Matters", 4),
    ("Deep Work", 5),
    ("Range", 4),
    ("Factfulness", 5),
    ("Invisible Women", 4),
    ("Born a Crime", 5),
    ("When Breath Becomes Air", 5),
    ("Into the Wild", 4),
    ("Into Thin Air", 4),
    ("The Wright Brothers", 4),
    ("Unbroken", 5),
    ("Seabiscuit", 4),
    ("The Boys in the Boat", 5),
    ("Killers of the Flower Moon", 4),
    ("The Devil in the White City", 4),
    ("Midnight in Chernobyl", 4),
    ("Radium Girls", 4),
    ("A Short History of Nearly Everything", 5)
]


In [ ]:
my_title_list = [("kingkiller", 5), ("The Lord of the Rings", 5), ("twilight", 3), ("eragon", 4), ("Harry Potter and the Sorcerer's Stone", 5), ("the mistborn",3),
                 ("brisingr", 4), ("a song of ice and fire", 5), ("how to train your dragon",3), ("percy jackson and the lightning thief",5), ("heroes of olympus", 3),
                 ("the 5th wave", 2), ("i am number four",4), ("pride and prejudice",1), ("The Forever War (The Forever War, #1)",1), ("The Smartest Guys in the Room",1),
                 ("The Mask of Command", 1), ("Endurance",1), ("Battle Cry of Freedom",1), ("Arkwright",1), ("fifty shades of grey",1)]

my_books = create_my_books_df(my_title_list_romance, vectorizer)
my_books

,user_id,book_id,title,rating
0,-1,5043803,"Twilight (Twilight, #1)",5
1,-1,871224,"Twilight (Twilight, #1)",5
2,-1,12024,"Twilight (Twilight, #1)",5
3,-1,3292087,"Twilight (Twilight, #1)",5
4,-1,108315,Twilight,5
...,...,...,...,...
245,-1,15717943,"Hopeless (Hopeless, #1)",4
246,-1,17156082,"Hopeless (Hopeless, #1)",4
247,-1,528560,A Hopeless Romantic,4
248,-1,17334411,"Hopeless (Hopeless, #1)",4


In [ ]:
csv_book_mapping = {}

with open ("book_id_map.csv", "r") as f:
  while True:
      line = f.readline()
      if not line:
          break
      csv_id, book_id = line.strip().split(",")

      csv_book_mapping[csv_id] = book_id

In [ ]:
# import zipfile

# valid_titles_set = set(titles["title"])
# ovelapping_users = {}

# with zipfile.ZipFile("goodreads_interactions.zip") as z:
#   file_name = z.namelist()[0]
#   with z.open(file_name) as f:
#     while True:
#         line = f.readline()
#         if not line:
#             break

#         line = line.decode('utf-8').strip()
#         user_id, csv_id, _, rating, _ = line.split(",")
#         book_id = csv_book_mapping.get(csv_id)

#         if book_id in valid_titles_set:
#           if user_id in ovelapping_users:
#             ovelapping_users[user_id] += 1
#           else:
#             ovelapping_users[user_id] = 1

In [ ]:
# len(ovelapping_users)

In [ ]:
# filtered_overlap_users = set([k for k in ovelapping_users if ovelapping_users[k] > my_books.shape[0]/15])

In [ ]:
# len(filtered_overlap_users)

In [ ]:
valid_book_id_set = set(titles["book_id"].unique())
len(valid_book_id_set)

495230

In [ ]:
overlap_users = {}
book_set = set(my_books["book_id"])

with zipfile.ZipFile("goodreads_interactions.zip") as z:
  file_name = z.namelist()[0]
  with z.open(file_name) as f:
    while True:
        line = f.readline()
        if not line:
            break

        line = line.decode('utf-8').strip()
        user_id, csv_id, _, rating, _ = line.split(",")
        book_id = csv_book_mapping.get(csv_id)
        if book_id is None:
            continue

        if book_id in book_set:
            if user_id not in overlap_users:
                overlap_users[user_id] = 1
            else:
                overlap_users[user_id] += 1

In [ ]:
len(overlap_users.keys())

457833

In [ ]:
filtered_overlap_users = set([k for k in overlap_users if overlap_users[k] > my_books.shape[0]/8])
len(filtered_overlap_users)

684

In [ ]:
interactions_list = []

with zipfile.ZipFile("goodreads_interactions.zip") as z:
  file_name = z.namelist()[0]
  with z.open(file_name) as f:
    while True:
        line = f.readline()
        if not line:
            break

        line = line.decode('utf-8').strip()
        user_id, csv_id, _, rating, _ = line.split(",")
        book_id = csv_book_mapping.get(csv_id)
        if book_id is None:
            continue

        if user_id in filtered_overlap_users and book_id in valid_book_id_set:
          interactions_list.append([user_id, book_id, rating])

In [ ]:
len(interactions_list)

4319606

In [ ]:
interactions_list[:10]

[['520', '13609836', '2'],
 ['520', '301082', '4'],
 ['520', '19501', '0'],
 ['520', '7654769', '4'],
 ['520', '10032672', '0'],
 ['520', '12232938', '0'],
 ['520', '7763', '0'],
 ['520', '12262741', '0'],
 ['520', '4325', '0'],
 ['520', '10959277', '0']]

In [ ]:
# check the csv_id for the matching book_id
csv_book_mapping['947']

'21'

In [ ]:
interactions = pd.DataFrame(interactions_list, columns=["user_id", "book_id", "rating"])
interactions.head()

,user_id,book_id,rating
0,520,13609836,2
1,520,301082,4
2,520,19501,0
3,520,7654769,4
4,520,10032672,0


In [ ]:
interactions.shape

(4319606, 3)

In [ ]:
interactions = pd.concat([my_books[['user_id', 'book_id', 'rating']], interactions], ignore_index=True)
interactions.head()

,user_id,book_id,rating
0,-1,5043803,5
1,-1,871224,5
2,-1,12024,5
3,-1,3292087,5
4,-1,108315,5


In [ ]:
interactions["book_id"] = interactions["book_id"].astype(str)
interactions["user_id"] = interactions["user_id"].astype(str)
interactions["rating"] = pd.to_numeric(interactions["rating"])

In [ ]:
interactions[interactions["user_id"] == '-1']

,user_id,book_id,rating
0,-1,5043803,5
1,-1,871224,5
2,-1,12024,5
3,-1,3292087,5
4,-1,108315,5
...,...,...,...
245,-1,15717943,4
246,-1,17156082,4
247,-1,528560,4
248,-1,17334411,4


In [ ]:
len(interactions["user_id"].unique())

685

In [ ]:
len(interactions["book_id"].unique())

366061

In [ ]:
def create_user_item_matrix(df):
    user_mapper = {user_id: i for i, user_id in enumerate(df['user_id'].unique())}
    book_mapper = {book_id: i for i, book_id in enumerate(df['book_id'].unique())}

    user_index = df['user_id'].map(user_mapper).values
    book_index = df['book_id'].map(book_mapper).values

    X = coo_matrix((df['rating'], (user_index, book_index)))

    return X.tocsr(), user_mapper, book_mapper


In [ ]:
X, user_mapper, book_mapper = create_user_item_matrix(interactions)

In [ ]:
X.shape

(685, 366061)

In [ ]:
n_total = X.shape[0]*X.shape[1]
n_ratings = X.nnz
sparsity = n_ratings/n_total
print(f"Matrix percentage filled: {round(sparsity*100,2)}% \nMatrix percentage empty: {100-round(sparsity*100,2)}%")

Matrix percentage filled: 1.72% 
Matrix percentage empty: 98.28%


In [ ]:
user_mapper["-1"]

0

In [ ]:
def find_similar_users(user_id, X, user_mapper, k=10, metric='cosine'):
    if user_id not in user_mapper:
        print(f"User ID {user_id} not found.")
        return []

    user_indice = user_mapper[user_id]
    user_vec = X[user_indice]

    kNN = NearestNeighbors(n_neighbors=k+1, metric=metric)
    kNN.fit(X)

    dist, neighbours_indices = kNN.kneighbors(user_vec, return_distance=True)
    neighbours_indices = neighbours_indices.flatten()
    dist = dist.flatten()
    indice_to_user_mapper = {v: k for k, v in user_mapper.items()}
    neighbours_user_ids = [indice_to_user_mapper[n] for n in neighbours_indices]

    neighbour = [id for id in neighbours_user_ids if id != user_id]
    print(dist)
    return neighbours_user_ids

In [ ]:
similar_users = find_similar_users('-1', X, user_mapper, k=100)
print(len(similar_users))
similar_users[:10]

[1.22124533e-15 8.70176375e-01 8.79416397e-01 8.79710241e-01
 8.85381009e-01 8.87036038e-01 8.88276372e-01 8.88348851e-01
 8.92026331e-01 8.92328155e-01 8.92665536e-01 8.94555763e-01
 8.94882940e-01 8.95290517e-01 8.96051024e-01 8.96078834e-01
 8.97192312e-01 8.97869975e-01 8.97958960e-01 8.98651918e-01
 8.99920436e-01 9.00542651e-01 9.00703417e-01 9.00874598e-01
 9.01673443e-01 9.01705006e-01 9.02093288e-01 9.02459971e-01
 9.02831828e-01 9.03679582e-01 9.04757633e-01 9.05290588e-01
 9.05349632e-01 9.05673901e-01 9.06237651e-01 9.06376396e-01
 9.06485403e-01 9.06501187e-01 9.06595757e-01 9.06744693e-01
 9.07839951e-01 9.08029349e-01 9.08438971e-01 9.09235535e-01
 9.09318534e-01 9.09767803e-01 9.09805008e-01 9.09876783e-01
 9.10544991e-01 9.10745068e-01 9.11335870e-01 9.12005618e-01
 9.12235895e-01 9.12296309e-01 9.12362572e-01 9.12665649e-01
 9.12712046e-01 9.13367757e-01 9.14553254e-01 9.15010692e-01
 9.15063176e-01 9.15202273e-01 9.15234537e-01 9.15681204e-01
 9.16557532e-01 9.168937

['-1',
 '299932',
 '95076',
 '193432',
 '70689',
 '394060',
 '354226',
 '110910',
 '155719',
 '269354']

In [ ]:
# def find_similar_users_cosine(user_id, X, user_mapper, k=10):
#     if user_id not in user_mapper:
#         print(f"User ID {user_id} not found.")
#         return []

#     user_index = user_mapper[user_id]
#     user_vec = X[user_index]

#     # Compute similarity to all users
#     similarity = cosine_similarity(user_vec, X).flatten()

#     # Set own similarity to -1 to exclude yourself
#     similarity[user_index] = -1

#     # Get top k users (highest similarity)
#     top_k_indices = np.argpartition(similarity, -k)[-k:]
#     # Sort by actual similarity score
#     top_k_indices = top_k_indices[np.argsort(similarity[top_k_indices])[::-1]]

#     # Map back to user_id
#     index_to_user = {v: k for k, v in user_mapper.items()}
#     similar_user_ids = [index_to_user[i] for i in top_k_indices]

#     return similar_user_ids

In [ ]:
def recommend_books_for_user(user_id, interactions_df, similar_users):
    # 1. Get all books you (the target user) have already rated
    user_books = my_books['book_id'].unique()

    # 2. Get all books rated by the similar users
    similar_users_books = interactions_df[interactions_df['user_id'].isin(similar_users)]
    print(len(similar_users_books))
    print(len(similar_users_books["user_id"].unique()))

    similar_users_books = similar_users_books[similar_users_books["user_id"] != "-1"]
    print(len(similar_users_books["user_id"].unique()))

    # 3. Remove books you've already rated (we only want new recommendations)
    recommendations = similar_users_books[~similar_users_books['book_id'].isin(user_books)]
    print(len(recommendations["book_id"].unique()))

    # 4. Aggregate recommendations by book
    book_recs = recommendations.groupby('book_id')['rating'].agg(['count', 'mean'])
    print(len(book_recs))
    book_recs = pd.merge(book_recs, titles, on="book_id", how="left")


    C = titles['ratings_count'].mean()
    m = titles['average_rating'].mean()
    book_recs['bayesian_score'] = (C * m + book_recs['ratings_count'] * book_recs['average_rating']) / (C + book_recs['ratings_count'])
    book_recs = book_recs.sort_values('bayesian_score', ascending=False)

    # book_recs["adjusted_count"] = book_recs["count"] * (book_recs["count"] / book_recs["ratings_count"])
    # book_recs["score"] = book_recs["mean"] * book_recs["adjusted_count"]
    # print(len(book_recs))


    return book_recs

In [ ]:
recommendations = recommend_books_for_user('-1', interactions, similar_users)
print("\nTop recommended books for you:")
recommendations.head(10)

149022
101
100
47373
47373

Top recommended books for you:


,book_id,count,mean,title,average_rating,ratings_count,url,cover_image,mod_title,bayesian_score
10393,17332218,1,0.000000,"Words of Radiance (The Stormlight Archive, #2)",4.77,78319,https://www.goodreads.com/book/show/17332218-w...,https://images.gr-assets.com/books/1507307927m...,words of radiance the stormlight archive 2,4.750484
45877,862041,9,2.666667,"Harry Potter Boxset (Harry Potter, #1-7)",4.74,193057,https://www.goodreads.com/book/show/862041.Har...,https://images.gr-assets.com/books/1392579059m...,harry potter boxset harry potter 17,4.732257
45042,8,2,4.500000,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...",4.77,34349,https://www.goodreads.com/book/show/8.Harry_Po...,https://s.gr-assets.com/assets/nophoto/book/11...,harry potter boxed set books 15 harry potter 15,4.726778
12550,17927395,49,2.551020,A Court of Mist and Fury (A Court of Thorns an...,4.71,120403,https://www.goodreads.com/book/show/17927395-a...,https://images.gr-assets.com/books/1485259138m...,a court of mist and fury a court of thorns and...,4.698110
1,10,2,2.500000,"Harry Potter Collection (Harry Potter, #1-6)",4.73,25245,https://www.goodreads.com/book/show/10.Harry_P...,https://images.gr-assets.com/books/1328867351m...,harry potter collection harry potter 16,4.674984
43940,7235533,2,0.000000,"The Way of Kings (The Stormlight Archive, #1)",4.64,151473,https://www.goodreads.com/book/show/7235533-th...,https://images.gr-assets.com/books/1507307887m...,the way of kings the stormlight archive 1,4.631363
47300,99298,2,5.000000,"The Harry Potter Collection 1-4 (Harry Potter,...",4.66,44587,https://www.goodreads.com/book/show/99298.The_...,https://s.gr-assets.com/assets/nophoto/book/11...,the harry potter collection 14 harry potter 14,4.630688
5323,136251,66,3.257576,Harry Potter and the Deathly Hallows (Harry Po...,4.62,1784684,https://www.goodreads.com/book/show/136251.Har...,https://images.gr-assets.com/books/1474171184m...,harry potter and the deathly hallows harry pot...,4.619280
29860,26073150,6,3.666667,A Court of Mist and Fury (A Court of Thorns an...,4.71,11129,https://www.goodreads.com/book/show/26073150-a...,https://images.gr-assets.com/books/1487592053m...,a court of mist and fury a court of thorns and...,4.597996
20933,22299763,25,1.400000,"Crooked Kingdom (Six of Crows, #2)",4.62,51205,https://www.goodreads.com/book/show/22299763-c...,https://images.gr-assets.com/books/1456172607m...,crooked kingdom six of crows 2,4.595738


In [ ]:
recommendations.shape

(47373, 10)

In [ ]:
recommendations.isnull().sum()

,0
book_id,0
count,0
mean,0
title,0
average_rating,0
ratings_count,0
url,0
cover_image,0
mod_title,0
bayesian_score,0


In [ ]:
titles.isnull().sum()

,0
book_id,0
title,0
average_rating,0
ratings_count,0
url,0
cover_image,0
mod_title,0


In [ ]:
interactions.isnull().sum()

,0
user_id,0
book_id,0
rating,0


In [ ]:
interactions[~interactions['book_id'].isin(titles['book_id'])].shape

(0, 3)

In [ ]:
# filter out similar books to my_books
def filter_recommendations_by_title_similarity(recommendations_df, my_books_df, vectorizer, threshold=0.5):
  my_books_df['mod_title'] = my_books_df['title'].str.replace("[^a-zA-Z0-9 ]", "", regex=True).str.lower().str.replace("\s+", " ", regex=True)
  my_titles_tfidf = vectorizer.transform(my_books_df['mod_title'].tolist())

  recommendations_tfidf = vectorizer.transform(recommendations_df['mod_title'].tolist())

  # my_books(row) x recommendations(column)
  similarity_matrix = cosine_similarity(my_titles_tfidf, recommendations_tfidf)

  # find the max similarity in each column
  max_similarity_per_recommendation = similarity_matrix.max(axis=0)

  # keeps the indices with the similarity below the threshold.
  filtered_indices = np.where(max_similarity_per_recommendation < threshold)[0]
  filtered_recommendations = recommendations_df.iloc[filtered_indices].copy()
  return filtered_recommendations.sort_values('bayesian_score', ascending=False)

In [ ]:
filtered_recommendations = filter_recommendations_by_title_similarity(
    recommendations,
    my_books,
    vectorizer,
    threshold=0.5  # You can adjust this, e.g., 0.4 for stricter or 0.6 for looser filtering
)

filtered_recommendations

,book_id,count,mean,title,average_rating,ratings_count,url,cover_image,mod_title,bayesian_score
10393,17332218,1,0.000000,"Words of Radiance (The Stormlight Archive, #2)",4.77,78319,https://www.goodreads.com/book/show/17332218-w...,https://images.gr-assets.com/books/1507307927m...,words of radiance the stormlight archive 2,4.750484
45877,862041,9,2.666667,"Harry Potter Boxset (Harry Potter, #1-7)",4.74,193057,https://www.goodreads.com/book/show/862041.Har...,https://images.gr-assets.com/books/1392579059m...,harry potter boxset harry potter 17,4.732257
45042,8,2,4.500000,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...",4.77,34349,https://www.goodreads.com/book/show/8.Harry_Po...,https://s.gr-assets.com/assets/nophoto/book/11...,harry potter boxed set books 15 harry potter 15,4.726778
12550,17927395,49,2.551020,A Court of Mist and Fury (A Court of Thorns an...,4.71,120403,https://www.goodreads.com/book/show/17927395-a...,https://images.gr-assets.com/books/1485259138m...,a court of mist and fury a court of thorns and...,4.698110
1,10,2,2.500000,"Harry Potter Collection (Harry Potter, #1-6)",4.73,25245,https://www.goodreads.com/book/show/10.Harry_P...,https://images.gr-assets.com/books/1328867351m...,harry potter collection harry potter 16,4.674984
...,...,...,...,...,...,...,...,...,...,...
15016,18465657,19,1.473684,The One & Only,3.11,37683,https://www.goodreads.com/book/show/18465657-t...,https://images.gr-assets.com/books/1387399626m...,the one only,3.148074
43559,6976,2,1.000000,The Mermaid Chair,3.10,63365,https://www.goodreads.com/book/show/6976.The_M...,https://images.gr-assets.com/books/1388259308m...,the mermaid chair,3.123364
3041,12712435,1,3.000000,Seating Arrangements,3.02,14377,https://www.goodreads.com/book/show/12712435-s...,https://images.gr-assets.com/books/1329425347m...,seating arrangements,3.123018
19589,2152,9,0.888889,The Jane Austen Book Club,3.07,53650,https://www.goodreads.com/book/show/2152.The_J...,https://images.gr-assets.com/books/1309282966m...,the jane austen book club,3.098453


In [ ]:
recommendations.shape, filtered_recommendations.shape

((47373, 10), (21290, 10))

In [ ]:
def create_user_item_matrix(df):
    user_mapper = {user_id: i for i, user_id in enumerate(df['user_id'].unique())}
    book_mapper = {book_id: i for i, book_id in enumerate(df['book_id'].unique())}

    user_index = df['user_id'].map(user_mapper).values
    book_index = df['book_id'].map(book_mapper).values

    X = coo_matrix((df['rating'], (user_index, book_index)))

    return X.tocsr(), user_mapper, book_mapper

# --- 2. Find similar books (Item-Based CF using item-user matrix)
def find_similar_books(book_id, X, book_mapper, k=10, metric='cosine'):
    item_user_matrix = X.T

    if book_id not in book_mapper:
        print(f"Book ID {book_id} not found.")
        return []

    book_index = book_mapper[book_id]
    book_vec = item_user_matrix[book_index]

    kNN = NearestNeighbors(n_neighbors=k+1, metric=metric)
    kNN.fit(item_user_matrix)

    dist, neighbours_indices = kNN.kneighbors(book_vec, return_distance=True)
    dist = dist.flatten()
    neighbours_indices = neighbours_indices.flatten()

    index_to_book = {v: k for k, v in book_mapper.items()}
    similar_books_with_scores = [
        (index_to_book[idx], 1 - d) for idx, d in zip(neighbours_indices, dist) if idx != book_index
    ]

    return similar_books_with_scores

# --- 3. Recommend books for user based on all their liked books (aggregate similar books)
def recommend_books_from_items(my_books_df, X, book_mapper, interactions_df):
    item_user_matrix = X.T
    index_to_book = {v: k for k, v in book_mapper.items()}

    all_similar_books = []

    for book_id in my_books_df['book_id']:
        if book_id not in book_mapper:
            print(f"Book ID {book_id} not found, skipping.")
            continue

        book_index = book_mapper[book_id]
        book_vec = item_user_matrix[book_index]

        kNN = NearestNeighbors(n_neighbors=50, metric='cosine')
        kNN.fit(item_user_matrix)

        neighbours_indices = kNN.kneighbors(book_vec, return_distance=False).flatten()

        similar_book_ids = [index_to_book[i] for i in neighbours_indices if i != book_index]

        all_similar_books.extend(similar_book_ids)

    # Aggregate by frequency
    from collections import Counter
    recommended_books_counter = Counter(all_similar_books)

    # Remove books already in my_books
    user_books_set = set(my_books_df['book_id'])
    recommended_books = [(book_id, count) for book_id, count in recommended_books_counter.items() if book_id not in user_books_set]

    # Create DataFrame
    recommendations_df = pd.DataFrame(recommended_books, columns=['book_id', 'similarity_count'])

    # Merge with interactions and titles for enrichment
    rec_books = interactions_df[interactions_df['book_id'].isin(recommendations_df['book_id'])]
    rec_books = rec_books.groupby('book_id')['rating'].agg(['count', 'mean']).reset_index()
    rec_books = pd.merge(rec_books, recommendations_df, on='book_id', how='left')
    rec_books = pd.merge(rec_books, titles, on="book_id", how="left")

    # Apply Bayesian score
    C = titles['ratings_count'].mean()
    m = titles['average_rating'].mean()
    rec_books['bayesian_score'] = (C * m + rec_books['ratings_count'] * rec_books['average_rating']) / (C + rec_books['ratings_count'])

    return rec_books.sort_values('bayesian_score', ascending=False)

In [ ]:
X, user_mapper, book_mapper = create_user_item_matrix(interactions)
print(f"User-Item matrix shape: {X.shape}")

User-Item matrix shape: (685, 366061)


In [ ]:
len(interactions["book_id"].unique())

366061

In [ ]:
recommendations_df = recommend_books_from_items(my_books, X, book_mapper, interactions)
print(f"Raw recommendations:\n{recommendations_df.head(10)}")

Raw recommendations:
       book_id  count      mean  similarity_count  \
1564  17332218     96  0.093750                 1   
2291  20150777      2  2.500000                 1   
879     136251    472  3.247881                15   
249   11221285     31  0.161290                 1   
3554  26073150     75  1.813333                 1   
4829    481749     16  0.312500                 1   
6131   9329354     36  0.138889                 1   
0            1    463  3.287257                14   
4858         5    462  3.471861                15   
5114         6    466  3.418455                17   

                                                  title  average_rating  \
1564     Words of Radiance (The Stormlight Archive, #2)            4.77   
2291     Words of Radiance (The Stormlight Archive, #2)            4.77   
879   Harry Potter and the Deathly Hallows (Harry Po...            4.62   
249   The Way of Kings, Part 2 (The Stormlight Archi...            4.78   
3554  A Court of Mis

In [ ]:
recommendations_df.head(30)

,book_id,count,mean,similarity_count,title,average_rating,ratings_count,url,cover_image,mod_title,bayesian_score
1564,17332218,96,0.093750,1,"Words of Radiance (The Stormlight Archive, #2)",4.77,78319,https://www.goodreads.com/book/show/17332218-w...,https://images.gr-assets.com/books/1507307927m...,words of radiance the stormlight archive 2,4.750484
2291,20150777,2,2.500000,1,"Words of Radiance (The Stormlight Archive, #2)",4.77,13717,https://www.goodreads.com/book/show/20150777-w...,https://images.gr-assets.com/books/1391535272m...,words of radiance the stormlight archive 2,4.669489
879,136251,472,3.247881,15,Harry Potter and the Deathly Hallows (Harry Po...,4.62,1784684,https://www.goodreads.com/book/show/136251.Har...,https://images.gr-assets.com/books/1474171184m...,harry potter and the deathly hallows harry pot...,4.619280
249,11221285,31,0.161290,1,"The Way of Kings, Part 2 (The Stormlight Archi...",4.78,7803,https://www.goodreads.com/book/show/11221285-t...,https://images.gr-assets.com/books/1314602075m...,the way of kings part 2 the stormlight archive 12,4.615990
3554,26073150,75,1.813333,1,A Court of Mist and Fury (A Court of Thorns an...,4.71,11129,https://www.goodreads.com/book/show/26073150-a...,https://images.gr-assets.com/books/1487592053m...,a court of mist and fury a court of thorns and...,4.597996
4829,481749,16,0.312500,1,Jesus the Christ,4.63,17364,https://www.goodreads.com/book/show/481749.Jes...,https://s.gr-assets.com/assets/nophoto/book/11...,jesus the christ,4.562043
6131,9329354,36,0.138889,1,"The Way of Kings, Part 1 (The Stormlight Archi...",4.67,10191,https://www.goodreads.com/book/show/9329354-th...,https://images.gr-assets.com/books/1357609842m...,the way of kings part 1 the stormlight archive 11,4.555413
0,1,463,3.287257,14,Harry Potter and the Half-Blood Prince (Harry ...,4.54,1713866,https://www.goodreads.com/book/show/1.Harry_Po...,https://images.gr-assets.com/books/1361039191m...,harry potter and the halfblood prince harry po...,4.539336
4858,5,462,3.471861,15,Harry Potter and the Prisoner of Azkaban (Harr...,4.53,1876252,https://www.goodreads.com/book/show/5.Harry_Po...,https://images.gr-assets.com/books/1499277281m...,harry potter and the prisoner of azkaban harry...,4.529403
5114,6,466,3.418455,17,Harry Potter and the Goblet of Fire (Harry Pot...,4.53,1792561,https://www.goodreads.com/book/show/6.Harry_Po...,https://images.gr-assets.com/books/1361482611m...,harry potter and the goblet of fire harry pott...,4.529375


In [ ]:
filtered_recommendations.head(30)

,book_id,count,mean,title,average_rating,ratings_count,url,cover_image,mod_title,bayesian_score
10393,17332218,1,0.000000,"Words of Radiance (The Stormlight Archive, #2)",4.77,78319,https://www.goodreads.com/book/show/17332218-w...,https://images.gr-assets.com/books/1507307927m...,words of radiance the stormlight archive 2,4.750484
45877,862041,9,2.666667,"Harry Potter Boxset (Harry Potter, #1-7)",4.74,193057,https://www.goodreads.com/book/show/862041.Har...,https://images.gr-assets.com/books/1392579059m...,harry potter boxset harry potter 17,4.732257
45042,8,2,4.500000,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...",4.77,34349,https://www.goodreads.com/book/show/8.Harry_Po...,https://s.gr-assets.com/assets/nophoto/book/11...,harry potter boxed set books 15 harry potter 15,4.726778
12550,17927395,49,2.551020,A Court of Mist and Fury (A Court of Thorns an...,4.71,120403,https://www.goodreads.com/book/show/17927395-a...,https://images.gr-assets.com/books/1485259138m...,a court of mist and fury a court of thorns and...,4.698110
1,10,2,2.500000,"Harry Potter Collection (Harry Potter, #1-6)",4.73,25245,https://www.goodreads.com/book/show/10.Harry_P...,https://images.gr-assets.com/books/1328867351m...,harry potter collection harry potter 16,4.674984
43940,7235533,2,0.000000,"The Way of Kings (The Stormlight Archive, #1)",4.64,151473,https://www.goodreads.com/book/show/7235533-th...,https://images.gr-assets.com/books/1507307887m...,the way of kings the stormlight archive 1,4.631363
47300,99298,2,5.000000,"The Harry Potter Collection 1-4 (Harry Potter,...",4.66,44587,https://www.goodreads.com/book/show/99298.The_...,https://s.gr-assets.com/assets/nophoto/book/11...,the harry potter collection 14 harry potter 14,4.630688
5323,136251,66,3.257576,Harry Potter and the Deathly Hallows (Harry Po...,4.62,1784684,https://www.goodreads.com/book/show/136251.Har...,https://images.gr-assets.com/books/1474171184m...,harry potter and the deathly hallows harry pot...,4.619280
29860,26073150,6,3.666667,A Court of Mist and Fury (A Court of Thorns an...,4.71,11129,https://www.goodreads.com/book/show/26073150-a...,https://images.gr-assets.com/books/1487592053m...,a court of mist and fury a court of thorns and...,4.597996
20933,22299763,25,1.400000,"Crooked Kingdom (Six of Crows, #2)",4.62,51205,https://www.goodreads.com/book/show/22299763-c...,https://images.gr-assets.com/books/1456172607m...,crooked kingdom six of crows 2,4.595738
